In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import parcels
import xarray as xr

In [ ]:
print(parcels.__version__)

In [ ]:
ds_fields = xr.open_zarr("../../elphe-hackathon_data/cmems_uovo_2001.zarr/")
ds_fields

In [ ]:
fields = {"U": ds_fields["uo"], "V": ds_fields["vo"]}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)
ds_fset = ds_fset.fillna(0.0)
ds_fset = ds_fset.isel(depth=slice(0, 2))
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)
print(fieldset)

In [ ]:
n_particles = 100_000

lon = np.random.uniform(-80, 20, size=(n_particles,))
lat = np.random.uniform(-35, 40, size=(n_particles,))
z = np.full_like(lon, ds_fields.depth.values[0])  # surface
time = np.array(
    [ds_fields.time.values[0] for _ in range(n_particles)]
)  # initial time of the input data

pset = parcels.ParticleSet(
    fieldset=fieldset.to_windowed_arrays(max_levels=2),
    # fieldset=fieldset,
    pclass=parcels.Particle,
    time=time,
    z=z,
    lat=lat,
    lon=lon,
)
print(pset)

In [ ]:
kernels = [parcels.kernels.AdvectionRK4]

In [ ]:
output_file = parcels.ParticleFile(
    "02_trajectories.parquet", outputdt=np.timedelta64(6, "h"), mode="w"
)

In [ ]:
pset.execute(
    kernels,
    runtime=np.timedelta64(9, "D"),
    dt=np.timedelta64(2, "h"),
    output_file=output_file,
)

In [ ]:
df = parcels.read_particlefile("02_trajectories.parquet")
df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
_df = (
    df.to_pandas()
    .sort_values("particle_id")
    .set_index("particle_id")
    .loc[range(0, 100_000, 1)]
)
scatter = ax.scatter(
    _df["lon"], _df["lat"], c=_df["time"], s=1, alpha=0.5, cmap="viridis_r"
)
ax.set_xlabel("Longitude [deg E]")
ax.set_ylabel("Latitude [deg N]")
ax.legend(loc="upper right")
fig.colorbar(scatter, ax=ax, label="time")
plt.show()